# College Event Management and Participation Analysis System
## Course: 25ITC31 - Foundations of Data Science
**Institution:** Velalar College of Engineering and Technology, Erode - 12  
**Department:** Information Technology  
**Academic Year:** 2026 - 2027 (ODD) | **Class:** 25IT3A&B  
**Student Register No.:** 31  
**Assigned Scenario:** Scenario 1 (College Event Management and Participation Analysis System)

---
### Syllabus Topics Covered:
- **Unit 1:** Data Science Fundamentals, Describing Data with Tables and Graphs, Mean, Median, Standard Deviation.
- **Unit 2:** Python Environment, Lists, Dictionaries, CSV/Database Handling, Module Imports.
- **Unit 3:** Data Wrangling with Pandas & NumPy, Data Indexing, Grouping, Aggregation.
- **Unit 4:** Data Cleaning & Transformation: Handling Missing Values, Duplicate Detection, Type Conversion, Merging/Joining DataFrames.
- **Unit 5:** Exploratory Data Analysis (EDA) & Data Visualization using Matplotlib and Seaborn (Pie, Bar, Boxplot, Countplot, Heatmap).

### Step 1: Import Required Libraries

In [1]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display and visual configurations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI'
plt.rcParams['font.family'] = 'sans-serif'
pd.set_option('display.max_columns', None)
print("Libraries successfully imported!")

### Step 2: Database Retrieval into Pandas DataFrames

In [2]:
db_path = "../database/event_management.db"
conn = sqlite3.connect(db_path)

# Read SQL tables directly into Pandas DataFrames
df_events = pd.read_sql_query("SELECT * FROM events", conn)
df_students = pd.read_sql_query("SELECT * FROM students", conn)
df_registrations = pd.read_sql_query("SELECT * FROM registrations", conn)
df_attendance = pd.read_sql_query("SELECT * FROM attendance", conn)
df_feedback = pd.read_sql_query("SELECT * FROM feedback", conn)

conn.close()
print(f"Extracted: {len(df_events)} Events, {len(df_students)} Students, {len(df_registrations)} Registrations")
df_events.head()

### Step 3: Data Cleaning, Preprocessing & Wrangling
- Converting string timestamps to DateTime objects
- Merging relational tables into a unified analytical DataFrame
- Handling missing values (null feedback comments and unrecorded attendance)

In [3]:
# 1. Type Conversions
df_events['event_date'] = pd.to_datetime(df_events['event_date'])
df_registrations['registration_date'] = pd.to_datetime(df_registrations['registration_date'])

# 2. Merging relational datasets
master_df = df_registrations.merge(df_events, on='event_id', how='left', suffixes=('_reg', '_event'))
master_df = master_df.merge(df_students, on='student_id', how='left', suffixes=('', '_student'))
master_df = master_df.merge(df_attendance, on='registration_id', how='left', suffixes=('', '_att'))
master_df = master_df.merge(df_feedback, on='registration_id', how='left', suffixes=('', '_fb'))

# 3. Handling Missing Values
master_df['status_att'] = master_df['status_att'].fillna('NOT_MARKED')
master_df['comments'] = master_df['comments'].fillna('No comments provided')
master_df['attended_binary'] = master_df['status_att'].apply(lambda x: 1 if x == 'PRESENT' else 0)

print("Master Analytical DataFrame Prepared successfully!")
master_df[['registration_id', 'event_name', 'name', 'department', 'status_att', 'rating']].head(10)

### Step 4: Descriptive Statistics & Summary Analysis

In [4]:
print("--- NUMERICAL ATTRIBUTES SUMMARY ---")
display(master_df[['max_capacity', 'rating', 'attended_binary']].describe())

valid_ratings = master_df['rating'].dropna()
print(f"\nOverall Mean Rating: {valid_ratings.mean():.2f} / 5.0")
print(f"Overall Turnout Rate: {(master_df['attended_binary'].sum() / len(master_df)) * 100:.2f}%")

### Step 5: Data Visualizations (Matplotlib & Seaborn)
#### Chart 1: Event Category Distribution (Pie Chart & Bar Chart)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
category_counts = df_events['category'].value_counts()
colors = sns.color_palette("Set2", len(category_counts))

axes[0].pie(category_counts, labels=category_counts.index, autopct='%1.1f%%',
            startangle=140, colors=colors, explode=[0.05]*len(category_counts),
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title("Event Distribution by Category", fontsize=13, fontweight='bold')

sns.barplot(x=category_counts.index, y=category_counts.values, ax=axes[1], palette="Blues_d")
axes[1].set_title("Number of Events per Category", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Count of Events")
for i, v in enumerate(category_counts.values):
    axes[1].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

#### Chart 2: Capacity vs. Confirmed Registrations

In [6]:
event_reg_summary = master_df.groupby(['event_id', 'event_name', 'max_capacity']).size().reset_index(name='registrations')
plt.figure(figsize=(12, 6))
bar_width = 0.35
indices = np.arange(len(event_reg_summary))

plt.bar(indices - bar_width/2, event_reg_summary['max_capacity'], width=bar_width,
        label='Max Capacity', color='#93c5fd', edgecolor='#2563eb')
plt.bar(indices + bar_width/2, event_reg_summary['registrations'], width=bar_width,
        label='Confirmed Registrations', color='#1d4ed8', edgecolor='#1e3a8a')

short_labels = [name[:20] + '...' if len(name) > 20 else name for name in event_reg_summary['event_name']]
plt.xticks(indices, short_labels, rotation=25, ha='right')
plt.title("Event Capacity vs. Confirmed Registrations (Reg No: 31)", fontsize=14, fontweight='bold')
plt.xlabel("Event Name")
plt.ylabel("Number of Seats")
plt.legend()
plt.tight_layout()
plt.show()

#### Chart 3: Department-wise Participation and Engagement

In [7]:
dept_counts = master_df['department'].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(y=dept_counts.index, x=dept_counts.values, palette="crest")
plt.title("Department-Wise Student Participation Volume", fontsize=14, fontweight='bold')
plt.xlabel("Total Registrations")
plt.ylabel("Department")
for i, v in enumerate(dept_counts.values):
    plt.text(v + 0.2, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

#### Chart 4: Attendance Rate by Event (% Present)

In [8]:
event_turnout = master_df.groupby('event_name').agg(
    registered=('registration_id', 'count'),
    attended=('attended_binary', 'sum')
).reset_index()
event_turnout['turnout_rate'] = (event_turnout['attended'] / event_turnout['registered']) * 100

plt.figure(figsize=(11, 5))
bars = sns.barplot(x='turnout_rate', y='event_name', data=event_turnout.sort_values(by='turnout_rate', ascending=False), palette="viridis")
plt.title("Event Attendance Conversion Rate (% Present / Registered)", fontsize=14, fontweight='bold')
plt.xlabel("Attendance Turnout Rate (%)")
plt.ylabel("Event Name")
plt.xlim(0, 115)
for p in bars.patches:
    width = p.get_width()
    bars.annotate(f"{width:.1f}%", (width + 1.5, p.get_y() + p.get_height() / 2.),
                  ha='left', va='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()

#### Chart 5: Feedback Rating Frequency & Category Comparison

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
valid_feedback = master_df.dropna(subset=['rating'])

sns.countplot(x='rating', data=valid_feedback, ax=axes[0], palette="YlOrRd")
axes[0].set_title("Frequency of Feedback Ratings (1 to 5 Stars)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")

sns.boxplot(x='category', y='rating', data=valid_feedback, ax=axes[1], palette="Pastel1")
axes[1].set_title("Rating Distribution by Event Category (Box Plot)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Event Category")
axes[1].set_ylabel("Rating (1-5)")

plt.tight_layout()
plt.show()

#### Chart 6: Multivariate Correlation Matrix Heatmap

In [10]:
agg_corr = master_df.groupby('event_id').agg(
    max_capacity=('max_capacity', 'first'),
    total_registered=('registration_id', 'count'),
    total_attended=('attended_binary', 'sum'),
    avg_rating=('rating', 'mean')
).reset_index()
agg_corr['attendance_pct'] = (agg_corr['total_attended'] / agg_corr['total_registered']) * 100
corr_matrix = agg_corr[['max_capacity', 'total_registered', 'total_attended', 'attendance_pct', 'avg_rating']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=1, linecolor='white')
plt.title("Multivariate Correlation Heatmap (Engagement Metrics)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()